# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 09 — Best Model Selection

---

### Purpose
Apply a principled, multi-criteria decision framework to rank all evaluated models
and select the single best model for production use. This notebook provides both
a quantitative ranking and a qualitative justification.

### Objectives
1. Load the master comparison table from Notebook 08
2. Apply a **weighted multi-criteria ranking** across all key metrics
3. Rank all models with a composite score
4. Identify the best model and provide a detailed analysis
5. Discuss the selected model's strengths and weaknesses
6. Explore the per-class performance of the best model
7. Analyse the most common misclassifications
8. Declare the finalist for comparison with the Transformer baseline

### Selection Criteria Weights
| Criterion | Weight | Rationale |
|---|---|---|
| Macro F1 | 35% | Most important — handles class imbalance fairly |
| Weighted F1 | 20% | Overall accuracy weighted by class frequency |
| ROC-AUC | 20% | Discriminative power across all thresholds |
| Prediction Speed | 15% | Deployment practicality |
| Memory Efficiency | 10% | Resource constraints |

### Notebook Outline
1. Imports
2. Configuration
3. Load Comparison Results
4. Multi-Criteria Ranking
5. Composite Score Ranking
6. Rank Visualisation
7. Best Model Deep-Dive
8. Per-Class Performance Analysis
9. Misclassification Analysis
10. Strengths and Weaknesses
11. Final Decision
12. Notebook Summary

---

## 1. Imports

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering.utils import load_feature_matrix, setup_logger
from src.visualization.plots import plot_confusion_matrix, plot_roc_curves
from src.utils.helpers import (
    set_global_seed, train_test_val_split, load_yaml, make_output_dirs,
    print_section_header, save_json,
)

print('✅ All libraries imported successfully.')

---

## 2. Configuration

In [ ]:
cfg = load_yaml(PROJECT_ROOT / 'configs' / 'training.yaml')

RANDOM_SEED = cfg['random_seed']
TEST_SIZE   = cfg['evaluation']['test_size']
VAL_SIZE    = cfg['evaluation']['val_size']
FEAT_CFG    = cfg['features']

set_global_seed(RANDOM_SEED)

DIR_MODELS  = PROJECT_ROOT / cfg['output']['models_dir']
DIR_FIGURES = PROJECT_ROOT / cfg['output']['figures_dir']
DIR_OUTPUTS = PROJECT_ROOT / cfg['output']['outputs_dir']
make_output_dirs(DIR_MODELS, DIR_FIGURES, DIR_OUTPUTS)

setup_logger(str(PROJECT_ROOT / cfg['logging']['log_file']), cfg['logging']['level'])

# ── Selection weights (must sum to 1.0) ───────────────────────────────────────
WEIGHTS = {
    'f1_macro':       0.35,
    'f1_weighted':    0.20,
    'roc_auc_macro':  0.20,
    'pred_speed':     0.15,   # derived from pred_time_s (lower time → higher score)
    'mem_efficiency': 0.10,   # derived from peak_memory_mb (lower MB → higher score)
}

print('Ranking weights:')
for metric, weight in WEIGHTS.items():
    print(f'  {metric:20s}: {weight:.0%}')

---

## 3. Load Comparison Results

In [ ]:
# ── Load master comparison table from Notebook 08 ─────────────────────────────
comparison_csv = DIR_OUTPUTS / 'model_comparison.csv'

if not comparison_csv.exists():
    raise FileNotFoundError(
        'model_comparison.csv not found. Run Notebook 08 first.'
    )

df = pd.read_csv(comparison_csv)
print(f'Loaded {len(df)} model configurations from master comparison table.')
print(f'Columns: {list(df.columns)}')
df.head()

---

## 4. Multi-Criteria Ranking

In [ ]:
# ── Normalise metrics to [0, 1] for composite scoring ─────────────────────────
ranked = df.copy()

def minmax_normalise(series: pd.Series) -> pd.Series:
    """Normalise a pandas Series to [0, 1] using min-max scaling."""
    rng = series.max() - series.min()
    return (series - series.min()) / rng if rng > 0 else pd.Series(np.ones(len(series)))

# Higher is better
ranked['norm_f1_macro']      = minmax_normalise(ranked['f1_macro'].fillna(0))
ranked['norm_f1_weighted']   = minmax_normalise(ranked['f1_weighted'].fillna(0))
ranked['norm_roc_auc_macro'] = minmax_normalise(ranked['roc_auc_macro'].fillna(0))

# Lower is better → invert for scoring
ranked['norm_pred_speed']    = minmax_normalise(-ranked['pred_time_s'].fillna(999))
ranked['norm_mem_efficiency']= minmax_normalise(-ranked['peak_memory_mb'].fillna(9999))

# ── Compute composite score ────────────────────────────────────────────────────
ranked['composite_score'] = (
    ranked['norm_f1_macro']       * WEIGHTS['f1_macro'] +
    ranked['norm_f1_weighted']    * WEIGHTS['f1_weighted'] +
    ranked['norm_roc_auc_macro']  * WEIGHTS['roc_auc_macro'] +
    ranked['norm_pred_speed']     * WEIGHTS['pred_speed'] +
    ranked['norm_mem_efficiency'] * WEIGHTS['mem_efficiency']
)

# Sort by composite score
ranked = ranked.sort_values('composite_score', ascending=False).reset_index(drop=True)
ranked.insert(0, 'composite_rank', range(1, len(ranked) + 1))

print('Composite Ranking:')
ranked[['composite_rank', 'model_name', 'feature_set',
        'f1_macro', 'f1_weighted', 'roc_auc_macro',
        'pred_time_s', 'composite_score']].head(10)

---

## 5. Composite Score Ranking

In [ ]:
# ── Full composite ranking table ──────────────────────────────────────────────
ranked[['composite_rank', 'model_name', 'feature_set',
        'f1_macro', 'roc_auc_macro', 'pred_time_s',
        'peak_memory_mb', 'composite_score']].style.highlight_max(
    subset=['f1_macro', 'composite_score'],
    color='lightgreen',
).highlight_min(
    subset=['composite_rank'],
    color='lightyellow',
).format({
    'f1_macro': '{:.4f}',
    'roc_auc_macro': '{:.4f}',
    'pred_time_s': '{:.4f}',
    'peak_memory_mb': '{:.2f}',
    'composite_score': '{:.4f}',
})

---

## 6. Rank Visualisation

In [ ]:
# ── Composite score bar chart ─────────────────────────────────────────────────
fig = px.bar(
    ranked.head(15),
    x='composite_score',
    y='model_name',
    orientation='h',
    color='composite_score',
    color_continuous_scale='Greens',
    title='Composite Score Ranking — Top 15 Models',
    template='plotly_dark',
    labels={'composite_score': 'Composite Score', 'model_name': 'Model'},
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

# ── Radar chart: top 5 models across key metrics ───────────────────────────────
radar_metrics = ['f1_macro', 'f1_weighted', 'roc_auc_macro', 'norm_pred_speed', 'norm_mem_efficiency']
radar_labels  = ['Macro F1', 'Weighted F1', 'ROC-AUC', 'Pred Speed', 'Mem Efficiency']

fig2 = go.Figure()
for _, row in ranked.head(5).iterrows():
    vals = [row.get(m, 0) for m in radar_metrics]
    fig2.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=radar_labels + [radar_labels[0]],
        name=str(row['model_name']),
        fill='toself',
        opacity=0.6,
    ))

fig2.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    title='Top-5 Models — Radar Chart (Normalised Metrics)',
    template='plotly_dark',
)
fig2.show()

---

## 7. Best Model Deep-Dive

In [ ]:
# ── Extract the #1 ranked model ───────────────────────────────────────────────
best_row = ranked.iloc[0]
BEST_MODEL_NAME  = str(best_row['model_name'])
BEST_FEATURE_SET = str(best_row['feature_set'])

print('=' * 60)
print(f'  🏆  SELECTED BEST MODEL')
print('=' * 60)
print(f'  Model         : {BEST_MODEL_NAME}')
print(f'  Feature Set   : {BEST_FEATURE_SET}')
print(f'  Composite Score: {best_row["composite_score"]:.4f}')
print(f'  Macro F1      : {best_row.get("f1_macro", "N/A")}')
print(f'  Weighted F1   : {best_row.get("f1_weighted", "N/A")}')
print(f'  ROC-AUC       : {best_row.get("roc_auc_macro", "N/A")}')
print(f'  Pred Time (s) : {best_row.get("pred_time_s", "N/A")}')
print('=' * 60)

---

## 8. Per-Class Performance Analysis

In [ ]:
# ── Re-load best model and evaluate per-class metrics ─────────────────────────
from sklearn.metrics import classification_report

# Map model name → saved path (update after running NB03–NB08)
# This is a template; the actual path depends on your NB08 model registry
MODEL_PATH_MAP = {
    'LR / TF-IDF':       DIR_MODELS / 'logistic_regression' / 'logistic_regression_tfidf.joblib',
    'LR-Tuned / TF-IDF': DIR_MODELS / 'tuned' / 'logistic_regression_tuned.joblib',
    'SVM / TF-IDF':      DIR_MODELS / 'linear_svm' / 'linear_svm_tfidf.joblib',
    'RF / Emb':          DIR_MODELS / 'random_forest' / 'random_forest_embedding.joblib',
    'RF-Tuned / Emb':    DIR_MODELS / 'tuned' / 'random_forest_tuned.joblib',
    'XGB / Emb':         DIR_MODELS / 'xgboost' / 'xgboost_embedding.joblib',
    'XGB-Tuned / Emb':   DIR_MODELS / 'tuned' / 'xgboost_tuned.joblib',
}

FEATURE_TEST_MAP = {
    'tfidf':     'X_te_tf',
    'char':      'X_te_ch',
    'style':     'X_te_st',
    'embedding': 'X_te_em',
}

classes = np.load(
    str(PROJECT_ROOT / 'data' / 'features' / 'tfidf' / 'classes_tfidf_fingerprint.npy'),
    allow_pickle=True,
)

# ── Load best model ────────────────────────────────────────────────────────────
if BEST_MODEL_NAME in MODEL_PATH_MAP and MODEL_PATH_MAP[BEST_MODEL_NAME].exists():
    best_payload = joblib.load(MODEL_PATH_MAP[BEST_MODEL_NAME])
    best_estimator = best_payload['model'] if isinstance(best_payload, dict) else best_payload
    print(f'Best model loaded from: {MODEL_PATH_MAP[BEST_MODEL_NAME].name}')
else:
    print(f'⚠️  Model path not found for "{BEST_MODEL_NAME}". Update MODEL_PATH_MAP above.')
    best_estimator = None

In [ ]:
# ── Per-class F1 bar chart ────────────────────────────────────────────────────
if best_estimator is not None:
    # Load appropriate test set
    _, _, X_te_tf, _, _, y_te_tf = train_test_val_split(
        *load_feature_matrix(
            PROJECT_ROOT / FEAT_CFG['tfidf']['fingerprint'],
            PROJECT_ROOT / FEAT_CFG['labels']['fingerprint'],
        ),
        test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
    )

    # Choose test set based on best feature set
    feat_test_map = {
        'tfidf': X_te_tf,
    }
    X_te_best = feat_test_map.get(BEST_FEATURE_SET, X_te_tf)
    y_te_best = y_te_tf

    y_pred_best = best_estimator.predict(X_te_best)

    from sklearn.metrics import f1_score
    f1_per_class = f1_score(y_te_best, y_pred_best, average=None, zero_division=0)

    per_class_df = pd.DataFrame({
        'LLM Model': list(classes),
        'F1 Score':  f1_per_class,
    }).sort_values('F1 Score', ascending=False)

    fig = px.bar(
        per_class_df,
        x='LLM Model',
        y='F1 Score',
        color='F1 Score',
        color_continuous_scale='Greens',
        title=f'Per-Class F1 Score — {BEST_MODEL_NAME}',
        template='plotly_dark',
    )
    fig.update_yaxes(range=[0, 1.05])
    fig.show()

    print('Full Classification Report:')
    print(classification_report(y_te_best, y_pred_best, target_names=classes, zero_division=0))

---

## 9. Misclassification Analysis

In [ ]:
# ── Identify most frequent misclassification pairs ────────────────────────────
if best_estimator is not None:
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(y_te_best, y_pred_best)

    # Off-diagonal entries are errors
    error_pairs = []
    for true_idx, row in enumerate(cm):
        for pred_idx, count in enumerate(row):
            if true_idx != pred_idx and count > 0:
                error_pairs.append({
                    'True Class':      classes[true_idx],
                    'Predicted Class': classes[pred_idx],
                    'Count':           count,
                })

    error_df = pd.DataFrame(error_pairs).sort_values('Count', ascending=False)

    print('Top Misclassification Pairs:')
    print(error_df.head(10).to_string(index=False))

    fig = px.bar(
        error_df.head(15),
        x='Count',
        y=error_df.head(15).apply(
            lambda r: f"{r['True Class']} → {r['Predicted Class']}", axis=1
        ),
        orientation='h',
        title='Most Common Misclassifications',
        template='plotly_dark',
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.show()

---

## 10. Strengths and Weaknesses

### Best Model Analysis

> **Note**: The analysis below is a template. Update with actual values after running the notebook.

#### Why This Model Was Selected

The selected model achieved the highest **composite score** by excelling on:
- **Macro F1** (weight 35%) — Equal treatment of all LLM classes
- **ROC-AUC** (weight 20%) — Strong discriminative power across all decision thresholds
- **Prediction speed** (weight 15%) — Suitable for real-time fingerprinting applications

#### Strengths
1. **Highest Macro F1** — Correctly identifies minority LLM classes
2. **Strong calibration** — Probability estimates are meaningful
3. **Interpretable** — Feature importance or coefficients reveal decision logic
4. **Low latency** — Sub-millisecond prediction on CPU
5. **Proven feature set** — Works on the feature type that preserves LLM fingerprints

#### Weaknesses
1. **Specific to feature set** — Requires pre-computed feature extraction pipeline
2. **No sequential context** — Does not model long-range dependencies (unlike Transformer)
3. **Distributional shift** — May degrade on future LLMs not seen during training
4. **Interpretability trade-off** — Ensemble models sacrifice linear interpretability

#### Limitations and Caveats
- Evaluation is performed on a held-out test split of the **synthetic dataset**.
  Real-world deployment may face domain shift.
- All models assume the set of possible LLM sources is **closed** (closed-set classification).
- The 384-dimensional embedding feature requires the sentence-transformers model at inference time.

---

## 11. Final Decision

In [ ]:
# ── Record the final selection ─────────────────────────────────────────────────
final_selection = {
    'selected_model':      BEST_MODEL_NAME,
    'feature_set':         BEST_FEATURE_SET,
    'composite_score':     float(best_row['composite_score']),
    'f1_macro':            float(best_row.get('f1_macro', 0)),
    'f1_weighted':         float(best_row.get('f1_weighted', 0)),
    'roc_auc_macro':       float(best_row.get('roc_auc_macro', 0)) if pd.notna(best_row.get('roc_auc_macro')) else None,
    'pred_time_s':         float(best_row.get('pred_time_s', 0)),
    'selection_rationale': (
        'Highest composite score under multi-criteria weighting: '
        '35% Macro F1, 20% Weighted F1, 20% ROC-AUC, 15% Prediction Speed, 10% Memory Efficiency.'
    ),
}

save_json(final_selection, DIR_OUTPUTS / 'best_model_selection.json')

print('\n' + '='*60)
print('  ✅  FINAL MODEL SELECTION RECORDED')
print('='*60)
for k, v in final_selection.items():
    print(f'  {k:25s}: {v}')
print('\nSaved → outputs/best_model_selection.json')

---

## 12. Notebook Summary

### Selection Process

| Step | Action | Result |
|---|---|---|
| 1 | Loaded master comparison table | All model metrics |
| 2 | Applied multi-criteria weighting | Composite score per model |
| 3 | Ranked by composite score | Ordered leaderboard |
| 4 | Deep-dive into #1 model | Per-class F1, misclassifications |
| 5 | Strengths & weaknesses analysis | Informed selection rationale |
| 6 | Recorded final selection | `outputs/best_model_selection.json` |

### What Comes Next

→ **Notebook 10**: Transformer Baseline (DistilBERT)
→ **Notebook 11**: Final Evaluation — classical ML vs Transformer, overall conclusions

### Research Significance
The selected model establishes the **classical ML baseline** for LLM fingerprinting.
It will be compared against the Transformer baseline in Notebook 11 to determine
whether deep learning offers a meaningful advantage for this task.

---
*Fingerprint Project — Best Model Selection — Complete*